In [17]:
import os
import numpy as np
import pandas as pd

# ============= SETTINGS =============
TRADING_DAYS = 256
LOOKBACK = 36
FORECAST_SCALER = 30.0
CAP = 20.0

BASE = r"C:\Users\loci_\Desktop\trading_webapp\DATA"
IN = os.path.join(BASE, "all_input_files")

# Instrument configs
INSTRUMENTS = {
    "AD1": {"file": os.path.join(IN, "AD1_small.csv"), "distance": 1/12},
    "RX1": {"file": os.path.join(IN, "RX1_small.csv"), "distance": 3/12},
}


# ============= HELPER FUNCTIONS =============
def ewma_variance(sq_returns: pd.Series, decay: float) -> pd.Series:
    """EWMA variance per your formula."""
    out = np.full(len(sq_returns), np.nan)
    vals = sq_returns.to_numpy()
    idx0 = np.argmax(~np.isnan(vals)) if np.any(~np.isnan(vals)) else None
    if idx0 is None:
        return pd.Series(out, index=sq_returns.index)
    out[idx0] = vals[idx0]
    prev = out[idx0]
    for i in range(idx0 + 1, len(vals)):
        x2 = 0.0 if np.isnan(vals[i]) else vals[i]
        prev = decay * x2 + (1 - decay) * prev
        out[i] = prev
    return pd.Series(out, index=sq_returns.index)


def compute_carry(df: pd.DataFrame, distance: float) -> pd.DataFrame:
    """Compute carry columns as per your spec."""
    df = df.copy()
    df["near"] = df["near"].astype(float)
    df["far"] = df["far"].astype(float)

    df["return"] = df["near"].diff()
    df["square_returns"] = df["return"] ** 2
    df["price_difference"] = df["far"] - df["near"]
    df["distance"] = distance
    df["net_expected_return"] = df["price_difference"] / df["distance"]

    decay = 2.0 / (LOOKBACK + 1.0)
    df["variance"] = ewma_variance(df["square_returns"], decay)
    df["standard_deviation"] = np.sqrt(df["variance"])
    df["annual_standard_deviation"] = df["standard_deviation"] * np.sqrt(TRADING_DAYS)

    df["raw_carry"] = df["net_expected_return"] / df["annual_standard_deviation"]
    df["forecast"] = df["raw_carry"] * FORECAST_SCALER
    df["capped_forecast"] = df["forecast"].clip(-CAP, CAP)
    df["forecast_x_return"] = df["capped_forecast"] * df["price_difference"].shift(-1)

    return df


def compute_sharpe(df: pd.DataFrame) -> float:
    """Annualised Sharpe of forecast_x_return."""
    s = df["forecast_x_return"].dropna()
    if len(s) < 2:
        return np.nan
    mu, sd = s.mean(), s.std()
    return np.nan if (sd == 0 or np.isnan(sd)) else (mu / sd) * np.sqrt(TRADING_DAYS)


# ============= MAIN RUNNER =============
for name, cfg in INSTRUMENTS.items():
    path = cfg["file"]
    dist = cfg["distance"]
    if not os.path.exists(path):
        print(f"❌ File not found: {path}")
        continue

    df = pd.read_csv(path)
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
        df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    df = compute_carry(df, distance=dist)
    sharpe = compute_sharpe(df)
    print(f"{name} carry Sharpe (raw forecast × next day's price diff): {sharpe:.3f}", flush=True)


AD1 carry Sharpe (raw forecast × next day's price diff): 22.903
RX1 carry Sharpe (raw forecast × next day's price diff): 22.401


In [1]:
python carry_run.py


SyntaxError: invalid syntax (2533064392.py, line 1)